# Phase 4 — Scorecard ASB workbench (PD Ins + PD Css)

Tune bins, review stability in charts, then fit scorecard

Work one product at a time (`PRODUCT`). Freeze bins before §3.

## Checklist
- [ ] §0 `SCORECARD_PARAMS` + `PRODUCT`
- [ ] §1 load + partition + slice
- [ ] §2 tuning loop until exit gate passes
- [ ] §3.1–3.3 RFE + Model_list
- [ ] §3.4–3.5 diagnose + subModel_list
- [ ] §3.5b BR review (subModel_list vars only)
- [ ] §3.6–3.8 pick winner + model_package
- [ ] §4 scorecard + calibration
- [ ] §5 Gini over time
- [ ] §6 save parquet artifacts
- [ ] Repeat for `css`

| Prof reference | Notebook |
|----------------|----------|
| Big_scorecard.xlsx | `big_scorecard` + plots |
| Gini_vars.xlsx | `gini_vars` |
| Variable_report.xlsx | `variable_report` + `plot_bin_stability` |
| Model_report.xlsx | `model_report` + `display_model_report` |


## §0 — Setup

In [25]:
import json
import pickle
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option("display.max_columns", None)

assert Path("../data/04_feature/abt_app.parquet").exists()
assert Path("../data/04_feature/decisions.parquet").exists()

PRODUCT = "css"

SCORECARD_PARAMS = {
    "target": "default12",
    "time_col": "period",
    "id_col": "aid",
    "train_end_period": "198512",
    "valid_start_period": "198601",
    "symbol_missing": "Missing",
    "symbol_other": "<OTHERS>",
    "woe_epsilon": 1e-4,
    "tree_random_state": 1234,
    "factor": 20 / np.log(2),
    "offset": 600 - (20 / np.log(2)) * np.log(50),
    "category_order": False,
    "candidate": {
        "prefixes": ["app", "act"],
    },
    "binning": {
        "ncategories_int": 3,
        "minimum_share_int": 0.05,
        "ncategories_nom": 3,
        "rare_threshold": 0.04,
    },
    "prescreen": {
        "iv_min": 0.02,
        "gini_min": 0.05,
        "psi_max": 0.1,
        "psi_tar_max": 0.1,
        "delta_gini_max": 0.2,
        "ar_diff_max": 0.20,
    },
    "stability": {
        "min_bin_n_period": 10,
        "min_bin_n_total": 30,
        "max_bad_rate_swing": 0.15,
        "min_periods_for_stability": 6,
        "max_plots": None,
    },
    "selection": {
        "number_vars": 12,
        "number_features": 5,
        "pvalue_max": 0.01,
        "vif_max": 3.0,
        "sort_metric": "gini_valid",
        "max_features": 12,
        "epsilon": 1e-4,
    },
}

# Flat aliases keep existing helper functions usable while tuning by step.
SCORECARD_PARAMS.update(SCORECARD_PARAMS["binning"])
SCORECARD_PARAMS.update(SCORECARD_PARAMS["prescreen"])
SCORECARD_PARAMS.update(SCORECARD_PARAMS["stability"])
SCORECARD_PARAMS.update(SCORECARD_PARAMS["selection"])
SCORECARD_PARAMS["features_to_plot_scope"] = "model_list"
SCORECARD_PARAMS["features_to_plot_final_only"] = True

## §1 — Data prep (run once)

In [2]:

LEAKAGE_COLS = [
    "default3", "default6", "default12", "decision", "decline_reason",
    "act_cus_active",
]
ID_COLS = ["cid", "aid", "period", "product"]


def partition_abt(df, train_end, valid_start, time_col="period"):
    work = df.copy()
    df_train = work[work[time_col] <= train_end]
    df_valid = work[work[time_col] >= valid_start]
    return df_train, df_valid


def prepare_target(df, target="default12"):
    out = df.copy()
    out[target] = out[target].map({".i": 0, ".d": 0, 0: 0, 1: 1})
    out = out.dropna(subset=[target])
    out[target] = out[target].astype(int)
    return out


def load_and_prepare_abt(abt_path, decisions_path):
    abt = pd.read_parquet(abt_path)
    decisions = pd.read_parquet(decisions_path)
    merged = abt.merge(
        decisions[["aid", "decision", "decline_reason"]],
        on="aid",
        how="left",
    )
    return prepare_target(merged)


def slice_product(df, product, decision="A"):
    return df[(df["product"] == product) & (df["decision"] == decision)].copy()


def get_candidate_features(df, product, params):
    work = slice_product(df, product)
    work = work.dropna(subset=["decision", "decline_reason"])
    prefixes = params.get("candidate", {}).get("prefixes", [])
    if prefixes:
        work = work[[c for c in work.columns if c[:3].lower() in prefixes or c in ID_COLS + LEAKAGE_COLS]]
    drop_cols = [c for c in LEAKAGE_COLS + ID_COLS if c in work.columns]
    work = work.drop(columns=drop_cols)
    numeric = [
        c for c in work.columns
        if pd.api.types.is_numeric_dtype(work[c]) and not pd.api.types.is_bool_dtype(work[c])
    ]
    nominal = [
        c for c in work.columns
        if c not in numeric
        and (
            pd.api.types.is_object_dtype(work[c])
            or pd.api.types.is_string_dtype(work[c])
            or isinstance(work[c].dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(work[c])
        )
    ]
    return {"numeric": numeric, "nominal": nominal, "all": numeric + nominal}

In [3]:
df_model = load_and_prepare_abt(
    "../data/04_feature/abt_app.parquet",
    "../data/04_feature/decisions.parquet",
)
assert df_model["aid"].is_unique
print(df_model.groupby(["product", "decision"])["default12"].mean())
print("rows:", len(df_model), "bad rate:", f"{df_model['default12'].mean():.2%}")

cand = get_candidate_features(df_model, PRODUCT, SCORECARD_PARAMS)
print(f"{PRODUCT} candidates:", len(cand["all"]))
print("candidate prefixes:", SCORECARD_PARAMS["candidate"]["prefixes"])

blocked_prefixes = ("agr", "ags")
blocked_found = [f for f in cand["all"] if f.startswith(blocked_prefixes)]
print("agr/ags present:", len(blocked_found))
print("act9_n_arrears in candidates:", "act9_n_arrears" in cand["all"])

df_accepted = df_model[df_model["decision"] == "A"].copy()
df_train, df_valid = partition_abt(
    df_accepted,
    SCORECARD_PARAMS["train_end_period"],
    SCORECARD_PARAMS["valid_start_period"],
    time_col=SCORECARD_PARAMS["time_col"],
)
train_product = slice_product(df_train, PRODUCT)
valid_product = slice_product(df_valid, PRODUCT)

for name, d in [("train", train_product), ("valid", valid_product)]:
    print(
        name,
        d["period"].min(), "→", d["period"].max(),
        "bad_rate", f"{d['default12'].mean():.2%}",
        "n", len(d),
    )

product  decision
css      A           0.683755
ins      A           0.137648
Name: default12, dtype: float64
rows: 44602 bad rate: 41.04%
css candidates: 56
candidate prefixes: ['app', 'act']
agr/ags present: 0
act9_n_arrears in candidates: True
train 197501 → 198612 bad_rate 69.00% n 20588
valid 198701 → 198712 bad_rate 60.75% n 1689


In [4]:
print(type(df_accepted['app_char_job_code'][0]))
print(df_accepted['app_char_job_code'][0])

<class 'str'>
Permanent


In [5]:
#Do we have mising values in target variable?
abt_app = pd.read_parquet("../data/04_feature/abt_app.parquet")
one=abt_app[SCORECARD_PARAMS["target"]]
one[one.isnull()==True].head()

13   NaN
29   NaN
31   NaN
32   NaN
38   NaN
Name: default12, dtype: float64

In [6]:
df_accepted[(df_accepted['cid'] == '0000001330') & (df_accepted['aid'] == 'css1975010100098')]['agr3_Mean_CMaxC_Days']

1    14.666667
Name: agr3_Mean_CMaxC_Days, dtype: float64

## §2 — Tuning loop (repeat until stable)

Re-run cells below after tuning `ncategories_int`, `minimum_share_int`, `ncategories_nom`, `psi_max`.

In [7]:
from sklearn.cluster import AgglomerativeClustering

def _bin_params(params):
    return {
        "other_label": params.get("symbol_other", "<OTHERS>"),
        "missing_label": params.get("symbol_missing", "Missing"),
        "max_bins": params.get("ncategories_int", 4),
        "min_bin_size": params.get("minimum_share_int", 0.03),
        "tree_random_state": params.get("tree_random_state", 1234),
        "rare_threshold": params.get("rare_threshold", params.get("minimum_share_unique", 0.03)),
        "max_groups": params.get("ncategories_nom", 4),
        "nominal_int_threshold": params.get("nominal_int_threshold", 10),
    }


def _numeric_interval_labels(feature, edges, missing_label):
  intervals = []
  for i in range(len(edges)):
    if i == 0 and edges[i] == -np.inf and edges[i + 1] != np.inf:
      intervals.append(f"{feature} < {edges[i + 1]}")
    elif i == 0 and edges[i] == -np.inf and edges[i + 1] == np.inf:
      intervals.append(f"{feature} <> {missing_label}")
    elif i > 0 and edges[i - 1] != -np.inf and edges[i] == np.inf:
      intervals.append(f"{edges[i - 1]} <= {feature}")
    elif (
      i > 0
      and edges[i - 1] != -np.inf
      and edges[i] != np.inf
      and edges[i] != missing_label
    ):
      intervals.append(f"{edges[i - 1]} <= {feature} < {edges[i]}")
    elif edges[i] == missing_label:
      intervals.append(f"{feature} = {missing_label}")
  return intervals


def _assign_numeric_bin(x, edges, intervals, missing_label, missing_bin):
  if pd.isna(x):
    return f"{missing_label}" if missing_bin else intervals[0]
  for i in range(len(edges)):
    if (
      i == 0
      and edges[i] == -np.inf
      and edges[i + 1] != np.inf
      and x < edges[i + 1]
    ):
      return intervals[0]
    if (
      i > 0
      and edges[i - 1] != -np.inf
      and edges[i] == np.inf
      and x >= edges[i - 1]
    ):
      return intervals[i - 1]
    if (
      i > 0
      and edges[i - 1] != -np.inf
      and edges[i] != np.inf
      and edges[i] != missing_label
      and edges[i - 1] <= x < edges[i]
    ):
      return intervals[i - 1]
  return intervals[-1]


def fit_bin_numeric(train, feature, target, params):
  cfg = _bin_params(params)
  miss_rate = train[feature].isna().mean()
  miss_share = 1 - miss_rate
  if miss_share <= 1e-5:
    miss_share = 1.0

  minimum_share = params["minimum_share_int"] / miss_share
  if minimum_share > 0.5:
    minimum_share = 0.5
  if minimum_share < params["minimum_share_int"]:
    minimum_share = params["minimum_share_int"]

  df_two = train[[target, feature]].dropna(subset=[feature])

  tree = DecisionTreeClassifier(
    max_leaf_nodes=cfg["max_bins"],
    min_weight_fraction_leaf=minimum_share,
    random_state=cfg["tree_random_state"],
  )
  tree.fit(df_two[feature].values.reshape(-1, 1), df_two[target])

  thresh = [round(s, 3) for s in tree.tree_.threshold if s != -2]
  edges = sorted([-np.inf, np.inf] + thresh)

  missing_bin = miss_rate > params["minimum_share_int"]
  if missing_bin:
    edges = edges + [cfg["missing_label"]]

  intervals = _numeric_interval_labels(feature, edges, cfg["missing_label"])

  return {
    "type": "numeric",
    "feature": feature,
    "edges": edges,
    "intervals": intervals,
    "missing_label": cfg["missing_label"],
    "missing_bin": missing_bin,
  }


from sklearn.cluster import AgglomerativeClustering

def fit_bin_nominal(train, feature, target, params):
  cfg = _bin_params(params)
  other_label = cfg["other_label"]
  missing_label = cfg["missing_label"]
  rare_threshold = cfg["rare_threshold"]
  max_groups = cfg["max_groups"]

  share = train.groupby(feature)[target].count() / train.shape[0]
  rate = train.groupby(feature)[target].mean()
  df_two = rate.to_frame(target).assign(share=share)
  df_two = df_two.loc[df_two["share"] > rare_threshold]

  n_clusters = min(max_groups, len(df_two))
  if n_clusters >= 2:
    labels = AgglomerativeClustering(
      n_clusters=n_clusters, metric="euclidean", linkage="ward"
    ).fit_predict(df_two[[target]])
    df_two["cluster"] = labels
  else:
    df_two["cluster"] = 0

  labsc = df_two[["cluster"]].copy()
  if df_two["share"].sum() < (1 - rare_threshold):
    labsc.loc[other_label] = -1
    labsc["cluster"] = labsc["cluster"] + 1
  labsc = labsc.sort_values("cluster").reset_index()

  category_map = {
    str(row[feature]): str(int(row["cluster"]))
    for _, row in labsc.iterrows()
    if row[feature] != other_label
  }
  for cat in train[feature].dropna().astype(str).unique():
    if cat not in category_map:
      category_map[cat] = other_label

  intervals = []
  current_cluster = None
  current_text = ""
  for _, row in labsc.iterrows():
    cat = str(row[feature])
    cl = int(row["cluster"])
    if current_cluster is None:
      current_text, current_cluster = cat, cl
    elif cl == current_cluster:
      current_text += f", {cat}"
    else:
      intervals.append(current_text)
      current_text, current_cluster = cat, cl
  if current_text:
    intervals.append(current_text)

  return {
    "type": "nominal",
    "feature": feature,
    "category_map": category_map,
    "intervals": intervals,
    "other_label": other_label,
    "missing_label": missing_label,
  }


def fit_binning_maps(train, features, target, params):
    cfg = _bin_params(params)
    nominal_int_threshold = cfg["nominal_int_threshold"]
    binning_maps = {}
    for feat in features:
        col = train[feat]
        is_nominal = (
            pd.api.types.is_bool_dtype(col)
            or pd.api.types.is_object_dtype(col)
            or pd.api.types.is_string_dtype(col)
            or isinstance(col.dtype, pd.CategoricalDtype)
            or (
                pd.api.types.is_integer_dtype(col)
                and col.nunique(dropna=True) <= nominal_int_threshold
            )
        )
        if is_nominal:
            binning_maps[feat] = fit_bin_nominal(train, feat, target, params)
        else:
            binning_maps[feat] = fit_bin_numeric(train, feat, target, params)
    return binning_maps


def apply_bins(df, binning_maps):
    base = df.drop(columns=[c for c in df.columns if c.endswith("_GRP")], errors="ignore")
    new_cols = {}
    for feat, spec in binning_maps.items():
        grp_col = f"{feat}_GRP"
        if spec["type"] == "numeric":
            new_cols[grp_col] = base[feat].apply(
                lambda x: _assign_numeric_bin(
                    x,
                    spec["edges"],
                    spec["intervals"],
                    spec["missing_label"],
                    spec["missing_bin"],
                )
            )
        elif spec["type"] == "nominal":
            cat_map = spec["category_map"]
            other_label = spec["other_label"]
            missing_label = spec["missing_label"]
            s = base[feat]
            missing_mask = s.isna()
            norm = s.astype("string")
            mapped = norm.map(cat_map).fillna(other_label)
            mapped.loc[missing_mask] = missing_label
            new_cols[grp_col] = mapped.astype("string")
        else:
            raise ValueError(f"Unknown binning type for {feat}: {spec['type']}")
    grp_df = pd.DataFrame(new_cols, index=base.index)
    return pd.concat([base, grp_df], axis=1).copy()

In [26]:
labsn = fit_bin_numeric(train_product, 'act_age', 'default12', SCORECARD_PARAMS)
labsn

{'type': 'numeric',
 'feature': 'act_age',
 'edges': [-inf, np.float64(61.5), np.float64(80.5), inf],
 'intervals': ['act_age < 61.5', '61.5 <= act_age < 80.5', '80.5 <= act_age'],
 'missing_label': 'Missing',
 'missing_bin': np.False_}

In [27]:
labsc = fit_bin_nominal(df_train, 'app_char_cars', 'default12', SCORECARD_PARAMS)
labsc

{'type': 'nominal',
 'feature': 'app_char_cars',
 'category_map': {'Owner': '0', 'No': '1'},
 'intervals': ['Owner', 'No'],
 'other_label': '<OTHERS>',
 'missing_label': 'Missing'}

In [13]:
def compute_gini(y_true, y_score):
    auc = roc_auc_score(y_true, y_score)
    auc = max(auc, 1 - auc)
    return float(np.clip(2 * auc - 1, 0.0, 1.0))


def compute_psi(train_series, valid_series, epsilon):
    train_dist = train_series.value_counts(normalize=True)
    valid_dist = valid_series.value_counts(normalize=True)
    all_bins = train_dist.index.union(valid_dist.index)
    train_pct = train_dist.reindex(all_bins, fill_value=0) + epsilon
    valid_pct = valid_dist.reindex(all_bins, fill_value=0) + epsilon
    psi_components = (train_pct - valid_pct) * np.log(train_pct / valid_pct)
    return float(max(psi_components.sum(), 0.0))


def check_vif(x):
    x_num = x.select_dtypes(include=[np.number]).copy()
    x_ = x_num.copy()
    x_.insert(0, "_intercept", 1.0)
    vif_values = {}
    for i, col in enumerate(x_.columns):
        if col == "_intercept":
            continue
        vif_values[col] = variance_inflation_factor(x_.values, i)
    return pd.Series(vif_values, name="vif")


def prescreen_features(train_woe, valid_woe, iv_table, params):
    target = params["target"]
    rows = []
    for _, row in iv_table.iterrows():
        feature = row["feature"]
        iv = row["iv"]
        woe_col = f"{feature}_WOE"
        grp_col = f"{feature}_GRP"
        reasons = []
        if iv < params["iv_min"]:
            reasons.append(f"IV < {params['iv_min']}")
        gini_train = gini_valid = ar_diff = np.nan
        if woe_col in train_woe.columns:
            gini_train = compute_gini(train_woe[target], train_woe[woe_col])
        else:
            reasons.append("No WOE column (train)")
        if woe_col in valid_woe.columns:
            gini_valid = compute_gini(valid_woe[target], valid_woe[woe_col])
        else:
            reasons.append("No WOE column (valid)")
        if not np.isnan(gini_train) and gini_train < params["gini_min"]:
            reasons.append(f"Gini train < {params['gini_min']}")
        if not np.isnan(gini_valid) and gini_valid < params["gini_min"]:
            reasons.append(f"Gini valid < {params['gini_min']}")
        if not np.isnan(gini_train) and not np.isnan(gini_valid):
            ar_diff = abs(gini_train - gini_valid)
            if ar_diff > params["ar_diff_max"]:
                reasons.append(f"AR-diff > {params['ar_diff_max']}")
        psi = np.nan
        if grp_col in train_woe.columns and grp_col in valid_woe.columns:
            psi = compute_psi(train_woe[grp_col], valid_woe[grp_col], params["woe_epsilon"])
            if psi > params["psi_max"]:
                reasons.append(f"PSI > {params['psi_max']}")
        else:
            reasons.append("No GRP column")
        rows.append({
            "feature": feature,
            "iv": iv,
            "gini_train": gini_train,
            "gini_valid": gini_valid,
            "ar_diff": ar_diff,
            "psi": psi,
            "status": "keep" if not reasons else "reject",
            "reason": "; ".join(reasons),
        })
    return pd.DataFrame(rows)

In [14]:
def build_woe_table(df_binned, feature_grp, target, epsilon):
    total_good = (df_binned[target] == 0).sum()
    total_bad = (df_binned[target] == 1).sum()
    grouped = (
        df_binned.groupby(feature_grp, observed=False)[target]
        .agg(n="count", bads="sum")
        .reset_index()
        .rename(columns={feature_grp: "bin"})
    )
    grouped["goods"] = grouped["n"] - grouped["bads"]
    grouped["dist_good"] = (grouped["goods"] + epsilon) / (total_good + epsilon)
    grouped["dist_bad"] = (grouped["bads"] + epsilon) / (total_bad + epsilon)
    grouped["woe"] = np.log(grouped["dist_good"] / grouped["dist_bad"])
    grouped["iv_component"] = (grouped["dist_good"] - grouped["dist_bad"]) * grouped["woe"]
    grouped["bad_rate"] = grouped["bads"] / grouped["n"]
    return grouped


def build_woe_maps(df_binned, grp_cols, target, epsilon):
    return {grp: build_woe_table(df_binned, grp, target, epsilon) for grp in grp_cols}


def compute_iv(woe_table):
    return float(max(woe_table["iv_component"].sum(), 0.0))


def build_iv_table(woe_maps):
    rows = []
    for grp, table in woe_maps.items():
        feature = grp[: -len("_GRP")]
        rows.append({"feature": feature, "iv": compute_iv(table)})
    return pd.DataFrame(rows).sort_values("iv", ascending=False).reset_index(drop=True)


def encode_woe(df_binned, woe_maps):
    base = df_binned.drop(columns=[c for c in df_binned.columns if c.endswith("_WOE")], errors="ignore")
    new_cols = {}
    for grp_col, woe_table in woe_maps.items():
        feature = grp_col[: -len("_GRP")]
        woe_col = f"{feature}_WOE"
        bin_to_woe = dict(zip(woe_table["bin"], woe_table["woe"]))
        new_cols[woe_col] = base[grp_col].map(bin_to_woe).fillna(0.0)
    woe_df = pd.DataFrame(new_cols, index=base.index)
    return pd.concat([base, woe_df], axis=1).copy()


def _bin_condition(binning_maps, feature, bin_label):
    spec = binning_maps.get(feature, {})
    if spec.get("type") == "numeric":
        return str(bin_label)
    if spec.get("type") == "nominal":
        inv = {v: k for k, v in spec.get("category_map", {}).items()}
        return inv.get(bin_label, str(bin_label))
    return str(bin_label)


def build_big_scorecard(train_binned, valid_binned, woe_maps, binning_maps, target, params):
    eps = params["woe_epsilon"]
    rows = []
    features = [grp[: -len("_GRP")] for grp in woe_maps]
    n_train = len(train_binned)
    n_valid = len(valid_binned)
    sum_bad_train = train_binned[target].sum()
    sum_bad_valid = valid_binned[target].sum()

    for feat in features:
        grp_col = f"{feat}_GRP"
        woe_tbl = woe_maps[grp_col]
        tr = (
            train_binned.groupby(grp_col, observed=False)[target]
            .agg(n_train="count", bads_train="sum")
            .reset_index()
            .rename(columns={grp_col: "bin"})
        )
        tr["goods_train"] = tr["n_train"] - tr["bads_train"]
        tr["bad_rate_train"] = tr["bads_train"] / tr["n_train"]
        tr["share_train"] = tr["n_train"] / n_train
        tr["bad_share_train"] = tr["bads_train"] / max(sum_bad_train, 1)

        va = (
            valid_binned.groupby(grp_col, observed=False)[target]
            .agg(n_valid="count", bads_valid="sum")
            .reset_index()
            .rename(columns={grp_col: "bin"})
        )
        va["goods_valid"] = va["n_valid"] - va["bads_valid"]
        va["bad_rate_valid"] = va["bads_valid"] / va["n_valid"]
        va["share_valid"] = va["n_valid"] / max(n_valid, 1)
        va["bad_share_valid"] = va["bads_valid"] / max(sum_bad_valid, 1)

        merged = tr.merge(va, on="bin", how="outer")
        woe_part = woe_tbl[["bin", "woe", "iv_component", "bad_rate"]].rename(
            columns={"bad_rate": "bad_rate_woe_train"}
        )
        merged = merged.merge(woe_part, on="bin", how="left")
        merged["variable"] = feat
        merged["condition"] = merged["bin"].map(lambda b: _bin_condition(binning_maps, feat, b))
        merged["psi_bin"] = (merged["share_train"] - merged["share_valid"]) * np.log(
            (merged["share_train"] + eps) / (merged["share_valid"] + eps)
        )
        merged["psi_bad"] = (merged["bad_share_train"] - merged["bad_share_valid"]) * np.log(
            (merged["bad_share_train"] + eps) / (merged["bad_share_valid"] + eps)
        )
        rows.append(merged)

    out = pd.concat(rows, ignore_index=True)
    col_order = [
        "variable", "bin", "condition",
        "n_train", "bads_train", "goods_train", "bad_rate_train", "share_train",
        "n_valid", "bads_valid", "goods_valid", "bad_rate_valid", "share_valid",
        "woe", "iv_component", "psi_bin", "psi_bad",
    ]
    return out[[c for c in col_order if c in out.columns]]


def build_gini_vars_table(
    train_binned, valid_binned, train_woe, valid_woe, big_scorecard, cand_features, target, params
):
    rows = []
    for feat in cand_features:
        woe_col = f"{feat}_WOE"
        gini_train = compute_gini(train_woe[target], train_woe[woe_col]) if woe_col in train_woe else np.nan
        gini_valid = compute_gini(valid_woe[target], valid_woe[woe_col]) if woe_col in valid_woe else np.nan
        delta = np.nan
        if gini_train and gini_train > 0:
            delta = abs(gini_train - gini_valid) / gini_train
        sub = big_scorecard[big_scorecard["variable"] == feat]
        iv = sub["iv_component"].sum() if len(sub) else 0.0
        psi = sub["psi_bin"].sum() if len(sub) else np.nan
        psi_tar = sub["psi_bad"].sum() if len(sub) else np.nan
        pm = train_binned[feat].isnull().mean() if feat in train_binned else np.nan
        nuniq = train_binned[feat].nunique(dropna=True) if feat in train_binned else np.nan
        rows.append({
            "variable": feat,
            "gini_train": gini_train,
            "gini_valid": gini_valid,
            "delta_gini": delta,
            "iv": iv,
            "psi": psi,
            "psi_tar": psi_tar,
            "percent_missing": pm,
            "count_unique": nuniq,
        })
    return pd.DataFrame(rows).sort_values("gini_train", ascending=False).reset_index(drop=True)


def apply_prescreen(gini_vars, params):
    return gini_vars.query(
        "gini_train > @params['gini_min']"
        " and delta_gini < @params['delta_gini_max']"
        " and psi_tar < @params['psi_tar_max']"
        " and psi < @params['psi_max']"
    )["variable"].tolist()

In [15]:
def bin_bad_rate_by_period(df_binned, feature, target, time_col="period"):
    grp_col = f"{feature}_GRP"
    g = (
        df_binned.groupby([grp_col, time_col], observed=False)[target]
        .agg(n="count", bads="sum")
        .reset_index()
        .rename(columns={grp_col: "bin", time_col: "period"})
    )
    g["goods"] = g["n"] - g["bads"]
    g["bad_rate"] = g["bads"] / g["n"]
    period_totals = df_binned.groupby(time_col).size().rename("period_n")
    g = g.merge(period_totals, left_on="period", right_index=True)
    g["share"] = g["n"] / g["period_n"]
    g["variable"] = feature
    return g[["variable", "bin", "period", "n", "bads", "goods", "bad_rate", "share"]]


def flag_unstable_bins(period_table, params):
    rows = []
    for (var, bin_label), sub in period_table.groupby(["variable", "bin"]):
        n_periods = sub["period"].nunique()
        min_n_period = sub["n"].min()
        total_n = sub["n"].sum()
        min_br = sub["bad_rate"].min()
        max_br = sub["bad_rate"].max()
        swing = max_br - min_br
        reasons = []
        if swing > params["max_bad_rate_swing"]:
            reasons.append(f"swing>{params['max_bad_rate_swing']}")
        if min_n_period < params["min_bin_n_period"]:
            reasons.append(f"min_n_period<{params['min_bin_n_period']}")
        if total_n < params["min_bin_n_total"]:
            reasons.append(f"total_n<{params['min_bin_n_total']}")
        rows.append({
            "variable": var,
            "bin": bin_label,
            "n_periods": n_periods,
            "min_bad_rate": min_br,
            "max_bad_rate": max_br,
            "bad_rate_swing": swing,
            "min_n_period": min_n_period,
            "total_n": total_n,
            "flag_unstable": bool(reasons),
            "flag_reason": "; ".join(reasons),
        })
    return pd.DataFrame(rows)


def bin_stability_report(df_binned, features, target, time_col, params):
    period_tables = {}
    parts = []
    for feat in features:
        pt = bin_bad_rate_by_period(df_binned, feat, target, time_col)
        period_tables[feat] = pt
        parts.append(pt)
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    flags = flag_unstable_bins(combined, params) if len(combined) else pd.DataFrame()
    return period_tables, flags


def build_variable_report(train_binned, big_scorecard, kept, target, time_col, params):
    report = {}
    period_tables, _ = bin_stability_report(
        train_binned, kept, target, time_col, params
    )
    static_cols = [
        "bin", "condition", "bad_rate_train", "share_train",
        "n_train", "bads_train", "goods_train",
    ]
    for feat in kept:
        static = big_scorecard.loc[
            big_scorecard["variable"] == feat, static_cols
        ].copy()
        period = period_tables.get(feat, pd.DataFrame())
        report[feat] = {"static": static, "period": period}
    return report

In [16]:
def plot_bin_stability(period_table, feature, train_end_period=None, min_n=None):
    if period_table.empty:
        return
    min_n = min_n or SCORECARD_PARAMS["min_bin_n_period"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Bin stability — {feature}")
    for bin_label, sub in period_table.groupby("bin"):
        sub = sub.sort_values("period")
        if sub["n"].min() < min_n:
            continue
        axes[0].plot(sub["period"], sub["bad_rate"], label=str(bin_label))
        axes[1].plot(sub["period"], sub["share"], label=str(bin_label))
    if train_end_period:
        for ax in axes:
            ax.axvline(train_end_period, color="gray", linestyle="--", alpha=0.7)
    axes[0].set_title("Bad rate over time")
    axes[0].set_ylabel("bad_rate")
    axes[1].set_title("Bin share over time")
    axes[1].set_ylabel("share")
    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
        ax.legend(fontsize=7, loc="best")
    plt.tight_layout()
    plt.show()


def plot_woe_ladder(woe_maps, big_scorecard, feature):
    grp = f"{feature}_GRP"
    sub = big_scorecard[big_scorecard["variable"] == feature].copy()
    if sub.empty:
        return
    sub = sub.sort_values("bad_rate_train")
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.bar(range(len(sub)), sub["bad_rate_train"].values, alpha=0.7, label="bad_rate_train")
    ax1.set_xticks(range(len(sub)))
    ax1.set_xticklabels(sub["bin"].astype(str), rotation=45, ha="right")
    ax1.set_ylabel("bad_rate_train")
    ax2 = ax1.twinx()
    ax2.plot(range(len(sub)), sub["woe"].values, color="red", label="WOE")
    ax2.set_ylabel("WOE")
    plt.title(f"WOE ladder — {feature}")
    plt.tight_layout()
    plt.show()


def plot_bin_bad_rate_bar(big_scorecard, variable):
    sub = big_scorecard[big_scorecard["variable"] == variable].sort_values("bad_rate_train")
    if sub.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(sub["bin"].astype(str), sub["bad_rate_train"])
    ax.set_title(f"Train bad rate by bin — {variable}")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


def plot_stability_heatmap(period_table, feature):
    if period_table.empty:
        return
    pivot = period_table.pivot(index="bin", columns="period", values="bad_rate")
    fig, ax = plt.subplots(figsize=(14, max(2, len(pivot) * 0.8)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=0.5)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index.astype(str))
    ax.set_xticks(range(0, len(pivot.columns), 6))
    ax.set_xticklabels(pivot.columns[::6], rotation=45, ha="right")
    ax.set_title(f"Bad rate heatmap — {feature}")
    plt.colorbar(im, ax=ax, label="bad_rate")
    plt.tight_layout()
    plt.show()
    

def display_variable_report(variable_report, feature):
    block = variable_report[feature]
    print(f"### {feature}")
    print("Static bins (Big_scorecard left block)")
    display(block["static"])
    print("Period × bin (Variable_report right block)")
    plot_bin_stability(
        block["period"],
        feature,
        train_end_period=SCORECARD_PARAMS["train_end_period"],
    )

In [28]:
# §2 — Tuning loop (re-run after changing params until exit gate passes)

binning_maps = fit_binning_maps(
    train_product, cand["all"], SCORECARD_PARAMS["target"], SCORECARD_PARAMS
)
train_binned = apply_bins(train_product, binning_maps)
valid_binned = apply_bins(valid_product, binning_maps)

grp_cols = [f"{f}_GRP" for f in cand["all"] if f"{f}_GRP" in train_binned.columns]
woe_maps = build_woe_maps(
    train_binned, grp_cols, SCORECARD_PARAMS["target"], SCORECARD_PARAMS["woe_epsilon"]
)
iv_table = build_iv_table(woe_maps)
train_woe = encode_woe(train_binned, woe_maps)
valid_woe = encode_woe(valid_binned, woe_maps)

big_scorecard = build_big_scorecard(
    train_binned, valid_binned, woe_maps, binning_maps,
    SCORECARD_PARAMS["target"], SCORECARD_PARAMS,
)
print("big_scorecard rows:", len(big_scorecard))
display(big_scorecard.head(10))

gini_vars = build_gini_vars_table(
    train_binned, valid_binned, train_woe, valid_woe,
    big_scorecard, cand["all"], SCORECARD_PARAMS["target"], SCORECARD_PARAMS,
)
display(gini_vars.head(15))

vars_selected = apply_prescreen(gini_vars, SCORECARD_PARAMS)
rejected = gini_vars[~gini_vars["variable"].isin(vars_selected)]
print(f"vars_selected: {len(vars_selected)} of {len(gini_vars)}")
display(rejected.head(20))

screen_report = prescreen_features(train_woe, valid_woe, iv_table, SCORECARD_PARAMS)

# Build artifacts now; BR-over-time plots are shown after Model_list in §3.
variable_report = build_variable_report(
    train_binned, big_scorecard, vars_selected,
    SCORECARD_PARAMS["target"], SCORECARD_PARAMS["time_col"], SCORECARD_PARAMS,
)
period_tables, stability_flags = bin_stability_report(
    train_binned, vars_selected, SCORECARD_PARAMS["target"],
    SCORECARD_PARAMS["time_col"], SCORECARD_PARAMS,
)

big_scorecard rows: 184


,variable,bin,condition,n_train,bads_train,goods_train,bad_rate_train,share_train,n_valid,bads_valid,goods_valid,bad_rate_valid,share_valid,woe,iv_component,psi_bin,psi_bad
0,act_age,61.5 <= act_age < 80.5,61.5 <= act_age < 80.5,10141,6909,3232,0.681294,0.492568,1026.0,619.0,407.0,0.603314,0.607460,0.040459,0.000812,0.024083,0.025205
1,act_age,80.5 <= act_age,80.5 <= act_age,1105,658,447,0.595475,0.053672,249.0,136.0,113.0,0.546185,0.147425,0.413537,0.009810,0.094619,0.090551
2,act_age,act_age < 61.5,act_age < 61.5,9342,6639,2703,0.710662,0.453759,414.0,271.0,143.0,0.654589,0.245115,-0.098416,0.004311,0.128452,0.115916
3,act_cc,0.634 <= act_cc < 0.885,0.634 <= act_cc < 0.885,6172,4591,1581,0.743843,0.299786,606.0,383.0,223.0,0.632013,0.358792,-0.265857,0.020058,0.010599,0.007224
4,act_cc,0.885 <= act_cc,0.885 <= act_cc,1448,1243,205,0.858425,0.070332,130.0,99.0,31.0,0.761538,0.076969,-1.002090,0.055492,0.000598,0.000879
5,act_cc,act_cc < 0.634,act_cc < 0.634,12968,8372,4596,0.645589,0.629881,953.0,544.0,409.0,0.570829,0.564239,0.200476,0.026227,0.007223,0.006247
6,act_loaninc,4.684 <= act_loaninc < 8.606,4.684 <= act_loaninc < 8.606,4931,3567,1364,0.723383,0.239508,482.0,310.0,172.0,0.643154,0.285376,-0.161120,0.006020,0.008034,0.009446
7,act_loaninc,8.606 <= act_loaninc,8.606 <= act_loaninc,2353,1955,398,0.830854,0.114290,222.0,167.0,55.0,0.752252,0.131439,-0.791510,0.059565,0.002395,0.004219
8,act_loaninc,act_loaninc < 4.684,act_loaninc < 4.684,13304,8684,4620,0.652736,0.646202,985.0,549.0,436.0,0.557360,0.583185,0.169095,0.019044,0.006465,0.010144
9,app_income,1067.5 <= app_income,1067.5 <= app_income,13304,8684,4620,0.652736,0.646202,985.0,549.0,436.0,0.557360,0.583185,0.169095,0.019044,0.006465,0.010144


,variable,gini_train,gini_valid,delta_gini,iv,psi,psi_tar,percent_missing,count_unique
0,act_ccss_maxdue,0.596833,0.591797,0.008438,1.489669,0.016335,0.005050,0.145425,8
1,act_ccss_dueutl,0.596315,0.601279,0.008325,1.520655,0.017482,0.003635,0.145425,80
2,act9_n_arrears,0.552769,0.554477,0.003089,1.281954,0.018466,0.003445,0.000000,10
3,act3_n_arrears,0.547849,0.559165,0.020655,1.344117,0.013447,0.000523,0.000000,4
4,act6_n_arrears,0.532331,0.538808,0.012169,1.291699,0.016505,0.001585,0.000000,7
5,act_ccss_min_pninst,0.526114,0.515123,0.020891,1.110170,0.018510,0.007139,0.145425,24
6,act12_n_arrears,0.513146,0.504052,0.017723,1.147169,0.013662,0.002235,0.000000,12
7,act_ccss_n_loans_act,0.446850,0.415241,0.070737,0.832791,0.006084,0.004681,0.145425,6
8,act_ccss_n_loan,0.441798,0.399953,0.094716,0.751645,0.003224,0.004260,0.000000,7
9,act_ccss_utl,0.413176,0.423597,0.025221,0.840648,0.026045,0.012811,0.145425,151


vars_selected: 33 of 56


,variable,gini_train,gini_valid,delta_gini,iv,psi,psi_tar,percent_missing,count_unique
20,act_ccss_seniority,0.237756,0.165977,0.301901,0.248980,0.121389,0.139918,0.076161,195
26,act_ccss_n_loans_hist,0.220262,0.215632,0.021019,0.271584,0.200019,0.192448,0.076161,49
30,act_ccss_n_statB,0.195349,0.180622,0.075389,0.140677,0.157785,0.200441,0.283029,43
31,act12_n_good_days,0.184611,0.129062,0.300895,0.150924,0.000175,0.000195,0.000000,12
33,act9_n_good_days,0.173848,0.134368,0.227096,0.122397,0.001291,0.001595,0.000000,10
34,act_cc,0.142117,0.101679,0.284539,0.101777,0.018419,0.014351,0.000000,3189
35,app_number_of_children,0.138979,0.096222,0.307648,0.087686,0.015152,0.014201,0.000000,4
38,act6_n_good_days,0.115065,0.080138,0.303540,0.075576,0.004858,0.002885,0.000000,7
41,act_cins_seniority,0.099717,0.027326,0.725967,0.052030,0.202554,0.217348,0.000000,203
42,act_cins_n_statC,0.079387,0.059642,0.248712,0.029197,0.036794,0.043417,0.134787,15


In [19]:
n_unstable = int(stability_flags["flag_unstable"].sum()) if len(stability_flags) else 0
thin_bins = big_scorecard.loc[
    (big_scorecard["variable"].isin(vars_selected))
    & (big_scorecard["share_train"] < SCORECARD_PARAMS["minimum_share_int"])
]
print(
    f"§2 check — PRODUCT={PRODUCT} vars_selected={len(vars_selected)} "
    f"unstable_bins={n_unstable} thin_bins={len(thin_bins)}"
)
if n_unstable == 0 and len(vars_selected) >= 10 and thin_bins.empty:
    print("§2 PASS — proceed to §3")
else:
    print("§2 NOT PASS — tune SCORECARD_PARAMS and re-run §2 block above")

§2 check — PRODUCT=css vars_selected=34 unstable_bins=147 thin_bins=8
§2 NOT PASS — tune SCORECARD_PARAMS and re-run §2 block above


## §3 — Model fit (after §2 stable)

Staged workflow — run cells in order; re-run from any checkpoint when tuning.

- **§3.1** Slice product data + candidate WOE columns
- **§3.2** RFE shortlist (top `number_vars`)
- **§3.3** Build `Model_list` (full shortlist + all `number_features` combos)
- **§3.4** Diagnose near-misses
- **§3.5** Filter `subModel_list`
- **§3.5b** BR stability review (`var_summary` + plots) — subModel_list variables only
- **§3.6** Pick winner (auto or manual)
- **§3.7** Fit final logit + effects
- **§3.8** Assemble `model_package` for §4+


In [20]:
# §3.1 — Slice product data + candidate WOE columns
target = SCORECARD_PARAMS["target"]
train_subset = slice_product(train_woe, PRODUCT)
valid_subset = slice_product(valid_woe, PRODUCT)
candidate_woe = [f"{f}_WOE" for f in vars_selected if f"{f}_WOE" in train_woe.columns]
print(f"PRODUCT={PRODUCT}  train={len(train_subset)}  valid={len(valid_subset)}  candidate_woe={len(candidate_woe)}")

PRODUCT=css  train=20588  valid=1689  candidate_woe=34


In [21]:
# §3.2 — RFE shortlist
number_vars = SCORECARD_PARAMS["selection"]["number_vars"]
rfe_model = LogisticRegression(solver="lbfgs", max_iter=1000)
rfe = RFE(estimator=rfe_model, n_features_to_select=1, step=1)
rfe.fit(train_subset[candidate_woe], train_subset[target])
rfe_ranking = pd.Series(rfe.ranking_, index=candidate_woe)
shortlisted = rfe_ranking[rfe_ranking <= number_vars].sort_values().index.tolist()
display(rfe_ranking.sort_values().head(15))
print("shortlisted:", shortlisted)

app_spendings_WOE              1
act_cins_dueutl_WOE            2
act_cins_n_loans_act_WOE       3
act_ccss_dueutl_WOE            4
act_cins_maxdue_WOE            5
act_ccss_n_statC_WOE           6
act_call_cc_WOE                7
app_income_WOE                 8
act_cins_min_seniority_WOE     9
act_loaninc_WOE               10
act_ccss_maxdue_WOE           11
act_ccss_n_loan_WOE           12
act_cins_min_pninst_WOE       13
act_cins_n_statC_WOE          14
act12_n_arrears_WOE           15
dtype: int64

shortlisted: ['app_spendings_WOE', 'act_cins_dueutl_WOE', 'act_cins_n_loans_act_WOE', 'act_ccss_dueutl_WOE', 'act_cins_maxdue_WOE', 'act_ccss_n_statC_WOE', 'act_call_cc_WOE', 'app_income_WOE', 'act_cins_min_seniority_WOE', 'act_loaninc_WOE', 'act_ccss_maxdue_WOE', 'act_ccss_n_loan_WOE']


In [22]:
# §3.3 — Build Model_list
number_features = SCORECARD_PARAMS["selection"]["number_features"]
rows = []
fit_failures = 0

def _assess_combo(feature_set):
    x_train = sm.add_constant(train_subset[feature_set], has_constant="add")
    y_train = train_subset[target]
    model = sm.Logit(y_train, x_train).fit(disp=0)
    feature_cols = [c for c in model.params.index if c != "const"]
    x_valid = sm.add_constant(valid_subset[feature_cols], has_constant="add")
    pred_train = model.predict(x_train)
    pred_valid = model.predict(x_valid)
    gini_train = compute_gini(train_subset[target], pred_train)
    gini_valid = compute_gini(valid_subset[target], pred_valid)
    pvalues = model.pvalues.drop(labels=["const"], errors="ignore")
    max_pvalue = float(pvalues.max()) if len(pvalues) else 0.0
    vif_series = check_vif(train_subset[feature_cols])
    max_vif = float(vif_series.max()) if len(vif_series) else 1.0
    betas = model.params.drop(labels=["const"], errors="ignore")
    nnegative_betas = int((betas < 0).sum())
    return {
        "Variables": ",".join(feature_set),
        "n_features": len(feature_set),
        "nnegative_betas": nnegative_betas,
        "max_pvalue": max_pvalue,
        "max_vif": max_vif,
        "gini_train": gini_train,
        "gini_valid": gini_valid,
        "ar_diff": abs(gini_train - gini_valid),
    }

try:
    rows.append(_assess_combo(shortlisted))
except Exception as exc:
    fit_failures += 1
    print("full shortlist fit failed:", exc)

for combo in combinations(shortlisted, number_features):
    try:
        rows.append(_assess_combo(list(combo)))
    except Exception:
        fit_failures += 1
        continue

Model_list = pd.DataFrame(rows)
print("Model_list shape:", Model_list.shape, "  fit_failures:", fit_failures)
display(Model_list.head(10))

Model_list shape: (925, 8)   fit_failures: 0


,Variables,n_features,nnegative_betas,max_pvalue,max_vif,gini_train,gini_valid,ar_diff
0,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",12,10,8.795269e-03,7091.500470,0.726177,0.722534,0.003643
1,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,1.447782e-11,12.137865,0.690781,0.690131,0.000651
2,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,5.546867e-11,12.140067,0.680133,0.676550,0.003583
3,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,2.006356e-10,12.136399,0.680974,0.670734,0.010240
4,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,7.545051e-10,12.138466,0.678804,0.676221,0.002583
5,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,2.004694e-10,12.136400,0.680918,0.670734,0.010183
6,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,7.057492e-05,18.701889,0.668743,0.659453,0.009290
7,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,5.021649e-12,12.147082,0.691369,0.683943,0.007426
8,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,9.793060e-46,5.136716,0.710608,0.715729,0.005120
9,"app_spendings_WOE,act_cins_dueutl_WOE,act_cins...",6,5,1.739918e-09,5.148142,0.698800,0.698159,0.000641


In [23]:
# §3.4 — Diagnose near-misses before filtering
sel = SCORECARD_PARAMS["selection"]
print("Model_list rows:", len(Model_list))
display(Model_list[["nnegative_betas", "max_pvalue", "max_vif", "gini_valid"]].describe())

print("pass sign:", (Model_list["nnegative_betas"] == 0).sum())
print("pass pvalue:", (Model_list["max_pvalue"] <= sel["pvalue_max"]).sum())
print("pass vif:", (Model_list["max_vif"] <= sel["vif_max"]).sum())
print(
    "pass all three:",
    (
        (Model_list["nnegative_betas"] == 0)
        & (Model_list["max_pvalue"] <= sel["pvalue_max"])
        & (Model_list["max_vif"] <= sel["vif_max"])
    ).sum(),
)

near = Model_list.sort_values(
    ["nnegative_betas", "max_pvalue", "max_vif", "gini_valid"],
    ascending=[True, True, True, False],
)
display(near.head(20))

Model_list rows: 925


,nnegative_betas,max_pvalue,max_vif,gini_valid
count,925.000000,9.250000e+02,925.000000,925.000000
mean,5.285405,1.676694e-01,1623.524518,0.645306
std,0.605399,2.880381e-01,2971.738551,0.079879
min,4.000000,1.233865e-68,1.374493,0.313626
25%,5.000000,7.011746e-12,4.429347,0.658705
50%,5.000000,2.593700e-05,12.116576,0.679211
75%,6.000000,2.767507e-01,19.470186,0.694391
max,10.000000,9.988566e-01,7091.500470,0.722534


pass sign: 0
pass pvalue: 576
pass vif: 207
pass all three: 0


,Variables,n_features,nnegative_betas,max_pvalue,max_vif,gini_train,gini_valid,ar_diff
574,"act_cins_dueutl_WOE,act_cins_n_loans_act_WOE,a...",6,4,6.979992e-10,7091.022727,0.407968,0.432416,0.024448
805,"act_cins_n_loans_act_WOE,act_cins_maxdue_WOE,a...",6,4,1.986203e-09,7090.889022,0.409978,0.426743,0.016765
578,"act_cins_dueutl_WOE,act_cins_n_loans_act_WOE,a...",6,4,1.294224e-04,7089.660090,0.555629,0.536968,0.018661
809,"act_cins_n_loans_act_WOE,act_cins_maxdue_WOE,a...",6,4,1.345668e-04,7089.508718,0.555973,0.531649,0.024323
731,"act_cins_n_loans_act_WOE,act_ccss_dueutl_WOE,a...",6,4,4.252704e-03,7089.508068,0.678813,0.671493,0.007321
500,"act_cins_dueutl_WOE,act_cins_n_loans_act_WOE,a...",6,4,4.298200e-03,7089.658942,0.681867,0.680654,0.001212
808,"act_cins_n_loans_act_WOE,act_cins_maxdue_WOE,a...",6,4,2.749894e-02,7089.512733,0.672558,0.659137,0.013421
770,"act_cins_n_loans_act_WOE,act_ccss_dueutl_WOE,a...",6,4,2.793680e-02,7090.623363,0.675100,0.676565,0.001465
577,"act_cins_dueutl_WOE,act_cins_n_loans_act_WOE,a...",6,4,2.909859e-02,7089.661059,0.674565,0.665727,0.008837
535,"act_cins_dueutl_WOE,act_cins_n_loans_act_WOE,a...",6,4,3.160325e-02,7089.659007,0.376601,0.376414,0.000188


In [ ]:
# §3.5 — Filter subModel_list (no raise)
sel = SCORECARD_PARAMS["selection"]
sort_metric = sel.get("sort_metric", "gini_valid")

subModel_list = Model_list[
    (Model_list["nnegative_betas"] == 0)
    & (Model_list["max_pvalue"] <= sel["pvalue_max"])
    & (Model_list["max_vif"] <= sel["vif_max"])
].sort_values(sort_metric, ascending=False).reset_index(drop=True)

print("subModel_list shape:", subModel_list.shape)
display(subModel_list.head(10))
if subModel_list.empty:
    print("No combo passes filters — inspect near-misses in §3.4, tune §2/params, or use MANUAL_OVERRIDE in §3.6")

subModel_list shape: (0, 8)


,Variables,n_features,nnegative_betas,max_pvalue,max_vif,gini_train,gini_valid,ar_diff


No combo passes filters — inspect near-misses in §3.4, tune §2/params, or use MANUAL_OVERRIDE in §3.6


In [ ]:
# §3.5b — BR stability review (subModel_list variables only)
if subModel_list.empty:
    print("subModel_list is empty — skip BR review until a passing combo exists or MANUAL_OVERRIDE is set in §3.6")
else:
    submodel_vars = sorted({
        (f[:-4] if f.endswith("_WOE") else f)
        for vars_str in subModel_list["Variables"]
        for f in vars_str.split(",")
    })
    print(f"subModel_list variables to review: {len(submodel_vars)}")

    flags_sub = stability_flags[stability_flags["variable"].isin(submodel_vars)]
    var_summary = (
        flags_sub.groupby("variable", as_index=False)
        .agg(
            n_bins=("bin", "count"),
            n_unstable=("flag_unstable", "sum"),
            max_swing=("bad_rate_swing", "max"),
            worst_bin=("bad_rate_swing", lambda s: flags_sub.loc[s.idxmax(), "bin"]),
            min_n_period=("min_n_period", "min"),
        )
        .sort_values(["n_unstable", "max_swing"], ascending=[False, False])
    )
    display(var_summary)

    problems = (
        flags_sub.query("flag_unstable")
        .sort_values("bad_rate_swing", ascending=False)
        [["variable", "bin", "bad_rate_swing", "min_n_period", "total_n", "flag_reason"]]
    )
    display(problems)

    max_plots = SCORECARD_PARAMS.get("max_plots")
    feats_to_plot = submodel_vars if max_plots is None else submodel_vars[:max_plots]
    for feat in feats_to_plot:
        if feat in variable_report:
            display_variable_report(variable_report, feat)


In [ ]:
# §3.6 — Pick winner (auto or manual)
WINNER_ROW = 0
# MANUAL_OVERRIDE = ["act9_n_arrears_WOE", "act_ccss_maxdue_WOE", ...]

# prof_css_6 = [
#     "act9_n_arrears",
#     "act_ccss_maxdue",
#     "act_ccss_n_statC",
#     "app_number_of_children",
#     "act_cc",
#     "act_age",
# ]

if "MANUAL_OVERRIDE" in dir() and MANUAL_OVERRIDE:
    selected_woe = list(MANUAL_OVERRIDE)
elif not subModel_list.empty:
    selected_woe = subModel_list.loc[WINNER_ROW, "Variables"].split(",")
else:
    selected_woe = []
    print("No auto winner — set MANUAL_OVERRIDE or relax filters in §3.5")

selected_features = [f[:-4] if f.endswith("_WOE") else f for f in selected_woe]
print("selected_woe:", selected_woe)
print("selected_features:", selected_features)


In [ ]:
# §3.7 — Fit final logit + effects
if not selected_woe:
    raise ValueError("No features selected — complete §3.6 with a winner or MANUAL_OVERRIDE")

x_train = sm.add_constant(train_subset[selected_woe], has_constant="add")
final_model = sm.Logit(train_subset[target], x_train).fit(disp=0)

feature_cols = [c for c in final_model.params.index if c != "const"]
x_valid = sm.add_constant(valid_subset[feature_cols], has_constant="add")
pred_train = final_model.predict(x_train)
pred_valid = final_model.predict(x_valid)
gini_train = compute_gini(train_subset[target], pred_train)
gini_valid = compute_gini(valid_subset[target], pred_valid)
pvalues = final_model.pvalues.drop(labels=["const"], errors="ignore")
max_pvalue = float(pvalues.max()) if len(pvalues) else 0.0
vif_series = check_vif(train_subset[feature_cols])
max_vif = float(vif_series.max()) if len(vif_series) else 1.0
betas = final_model.params.drop(labels=["const"], errors="ignore")
nnegative_betas = int((betas < 0).sum())

metrics = {
    "gini_train": gini_train,
    "gini_valid": gini_valid,
    "ar_diff": abs(gini_train - gini_valid),
    "max_pvalue": max_pvalue,
    "max_vif": max_vif,
    "nnegative_betas": nnegative_betas,
    "n_features": len(feature_cols),
    "pvalues": pvalues.to_dict(),
    "vif": vif_series.to_dict(),
}

print("metrics:", metrics)

effects = pd.DataFrame([
    {
        "feature": feat,
        "beta": final_model.params[feat],
        "pvalue": final_model.pvalues[feat],
        "vif": vif_series.get(feat, np.nan),
    }
    for feat in feature_cols
]).assign(abs_beta=lambda d: d["beta"].abs()).sort_values("abs_beta", ascending=False).drop(columns=["abs_beta"])
display(effects)


In [ ]:
# §3.8 — Assemble model_package for §4+
woe_tables = {}
for woe_col in selected_woe:
    raw_feat = woe_col[: -len("_WOE")]
    grp_key = f"{raw_feat}_GRP"
    if grp_key in woe_maps:
        woe_tables[woe_col] = woe_maps[grp_key]

model_package = {
    "product": PRODUCT,
    "features": selected_woe,
    "model": final_model,
    "metrics": metrics,
    "effects": effects,
    "train_subset": train_subset,
    "valid_subset": valid_subset,
    "woe_tables": woe_tables,
    "id_col": SCORECARD_PARAMS.get("id_col", "aid"),
    "rfe_ranking": rfe_ranking,
    "Model_list": Model_list,
    "subModel_list": subModel_list,
}

from math import comb
shortlisted_len = int((rfe_ranking <= SCORECARD_PARAMS["selection"]["number_vars"]).sum())
nf = SCORECARD_PARAMS["selection"]["number_features"]
expected_combo_rows = comb(shortlisted_len, nf) + 1 if shortlisted_len >= nf else 0

print("Model_list rows:", len(Model_list), "expected from shortlist:", expected_combo_rows)
print("contains agr/ags in candidates:", any(f.startswith(("agr", "ags")) for f in cand["all"]))
print("act9_n_arrears in candidates:", "act9_n_arrears" in cand["all"])
print("notebook selected vars:", selected_features)
print("model_package ready for §4")


## §4 — Scorecard points & calibration

In [ ]:
def scale_scorecard(model_package, factor, offset):
    model = model_package["model"]
    features = model_package["features"]
    intercept = model.params.get("const", 0.0)
    base_points = offset - factor * intercept
    rows = []
    for feat in features:
        beta = model.params[feat]
        woe_table = model_package.get("woe_tables", {}).get(feat)
        if woe_table is None:
            rows.append({"feature": feat, "bin": "<ALL>", "woe": np.nan, "points": np.nan})
            continue
        for _, row in woe_table.iterrows():
            rows.append({
                "feature": feat,
                "bin": row["bin"],
                "woe": row["woe"],
                "points": -beta * row["woe"] * factor,
            })
    points_table = pd.DataFrame(rows)
    points_table.attrs["base_points"] = base_points
    points_table.attrs["intercept"] = intercept
    points_table.attrs["factor"] = factor
    points_table.attrs["offset"] = offset
    return points_table


def score_applicants(df_woe, model_package, points_table=None):
    features = model_package["features"]
    id_col = model_package.get("id_col", "aid")
    out = pd.DataFrame({id_col: df_woe[id_col].values})
    if points_table is not None:
        base_points = points_table.attrs.get("base_points", 0.0)
        total = pd.Series(base_points, index=df_woe.index, dtype=float)
        for feat in features:
            raw_feat = feat[: -len("_WOE")]
            grp_col = f"{raw_feat}_GRP"
            feat_points_map = (
                points_table[points_table["feature"] == feat]
                .set_index("bin")["points"]
                .to_dict()
            )
            contrib = df_woe[grp_col].map(feat_points_map).fillna(0.0)
            out[f"{feat}_points"] = contrib.values
            total = total + contrib
        out["score"] = total.values
    else:
        model = model_package["model"]
        x = df_woe[features].copy()
        x.insert(0, "const", 1.0)
        out["score"] = x.values @ model.params[["const"] + features].values
    return out.rename(columns={id_col: "aid"}) if id_col != "aid" else out


def calibrate_pd(scores_df, target="default12"):
    y = scores_df[target].values
    score = scores_df["score"].values
    x = sm.add_constant(score)
    calib_model = sm.Logit(y, x).fit(disp=0)
    param_index = calib_model.params.index.tolist()
    a = float(calib_model.params["const"]) if "const" in param_index else float(calib_model.params.iloc[0])
    slope_cols = [c for c in param_index if c != "const"]
    b = float(calib_model.params[slope_cols[0]]) if slope_cols else 0.0
    pd_calibrated = calib_model.predict(x)
    return {
        "params": {"intercept": a, "coef": b, "target": target, "score_col": "score"},
        "diagnostics": {
            "auc_before": float(roc_auc_score(y, score)),
            "auc_after": float(roc_auc_score(y, pd_calibrated)),
            "brier_before": float(brier_score_loss(y, (score - score.min()) / (score.max() - score.min() + 1e-12))),
            "brier_after": float(brier_score_loss(y, pd_calibrated)),
            "mean_pd_predicted": float(pd_calibrated.mean()),
            "mean_pd_actual": float(y.mean()),
        },
    }


def variable_importance_from_points(points_table):
    scale = (
        points_table.groupby("feature")["points"]
        .agg(min_points="min", max_points="max")
        .reset_index()
    )
    scale["range"] = scale["max_points"] - scale["min_points"]
    total_range = scale["range"].sum()
    scale["importance_pct"] = scale["range"] / total_range if total_range else 0.0
    return scale.sort_values("importance_pct", ascending=False)


def plot_calibration_curve(scores_df, target, n_deciles=10):
    work = scores_df[[target, "score"]].dropna().copy()
    work["decile"] = pd.qcut(work["score"], n_deciles, duplicates="drop")
    cal = work.groupby("decile", observed=False).agg(
        n=("score", "count"),
        mean_score=("score", "mean"),
        bad_rate=(target, "mean"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(cal["mean_score"], cal["bad_rate"])
    ax.set_xlabel("mean score (decile)")
    ax.set_ylabel("actual bad rate")
    ax.set_title("Calibration curve")
    plt.tight_layout()
    plt.show()
    return cal


In [ ]:
points_table = scale_scorecard(
    model_package, SCORECARD_PARAMS["factor"], SCORECARD_PARAMS["offset"]
)
display(points_table.head(20))

importance = variable_importance_from_points(points_table)
display(importance)
importance.plot.barh(x="feature", y="importance_pct", legend=False, figsize=(8, 5))
plt.xlabel("importance_pct")
plt.tight_layout()
plt.show()

valid_scored = score_applicants(valid_woe, model_package, points_table)
valid_scored = valid_scored.merge(
    valid_product[[SCORECARD_PARAMS["id_col"], SCORECARD_PARAMS["target"]]].rename(
        columns={SCORECARD_PARAMS["id_col"]: "aid"}
    ),
    on="aid",
)
calibration = calibrate_pd(valid_scored, target=SCORECARD_PARAMS["target"])
print("calibration params:", calibration["params"])

cal_table = plot_calibration_curve(valid_scored, SCORECARD_PARAMS["target"])
display(cal_table)

## §5 — Model stability (Gini over time)

In [ ]:
def gini_over_time(scored_df, target, time_col="period", score_col="score", min_n=30):
    rows = []
    for period, sub in scored_df.groupby(time_col):
        if len(sub) < min_n:
            continue
        g = compute_gini(sub[target], sub[score_col])
        rows.append({
            "period": period,
            "n": len(sub),
            "bad_rate": sub[target].mean(),
            "gini": g,
        })
    return pd.DataFrame(rows).sort_values("period")


def plot_gini_over_time(gini_by_period, overall_gini=None):
    if gini_by_period.empty:
        return
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(gini_by_period["period"], gini_by_period["gini"])
    median_g = gini_by_period["gini"].median()
    ax.axhline(median_g, color="gray", linestyle="--", label=f"median {median_g:.2%}")
    if overall_gini is not None:
        ax.axhline(overall_gini, color="green", linestyle=":", label=f"overall valid {overall_gini:.2%}")
    ax.set_title("Gini over time")
    ax.set_ylabel("Gini")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
    plt.tight_layout()
    plt.show()
    flagged = gini_by_period[gini_by_period["gini"] < median_g - 0.10]
    if len(flagged):
        print("Warning: periods with Gini >10pp below median:")
        display(flagged)

In [ ]:
scored_with_period = valid_scored.merge(
    valid_product[[SCORECARD_PARAMS["id_col"], SCORECARD_PARAMS["time_col"]]].rename(
        columns={SCORECARD_PARAMS["id_col"]: "aid"}
    ),
    on="aid",
)
gini_time = gini_over_time(
    scored_with_period,
    SCORECARD_PARAMS["target"],
    SCORECARD_PARAMS["time_col"],
)
display(gini_time)
plot_gini_over_time(gini_time, overall_gini=model_package["metrics"]["gini_valid"])

## §6 — Save artifacts

In [ ]:
def build_model_report(model_package, points_table, gini_time, calibration, cal_table, gini_vars):
    m = model_package["metrics"]
    main = pd.DataFrame({
        "Measure": [
            "gini_train", "gini_valid", "ar_diff", "max_vif",
            "max_pvalue", "n_features", "nnegative_betas",
        ],
        "Value": [
            m["gini_train"], m["gini_valid"], m["ar_diff"], m["max_vif"],
            m["max_pvalue"], m["n_features"], m["nnegative_betas"],
        ],
    })
    effects = model_package.get("effects")
    if effects is None:
        feature_cols = model_package["features"]
        vif_series = check_vif(model_package["train_subset"][feature_cols])
        effects = pd.DataFrame([
            {"feature": feat, "beta": model_package["model"].params[feat],
             "pvalue": model_package["model"].pvalues[feat],
             "vif": vif_series.get(feat, np.nan)}
            for feat in feature_cols
        ])
    importance = variable_importance_from_points(points_table)
    scorecard = points_table.copy()
    scorecard["points_round"] = scorecard["points"].round(0)
    cal_df = pd.DataFrame([calibration["params"]])
    cal_df = pd.concat([cal_df, cal_table], ignore_index=True)
    final_vars = [f[: -len("_WOE")] for f in model_package["features"]]
    gini_final = gini_vars[gini_vars["variable"].isin(final_vars)]
    return {
        "Main_measures": main,
        "Effects": effects,
        "Gini_over_time": gini_time,
        "Scorecard": scorecard,
        "Variable importance": importance,
        "Calibration": cal_df,
        "Gini_vars_final": gini_final,
    }


def display_model_report(model_report):
    for sheet_name, df in model_report.items():
        print(f"## {sheet_name}")
        display(df)
        if sheet_name == "Gini_over_time":
            plot_gini_over_time(df)
        elif sheet_name == "Variable importance":
            df.plot.barh(x="feature", y="importance_pct", legend=False, figsize=(8, 4))
            plt.tight_layout()
            plt.show()


def save_product_artifacts(
    product, out_dir, big_scorecard, gini_vars, stability_flags, gini_time,
    model_package, points_table, screen_report, woe_maps, calibration,
):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    big_scorecard.to_parquet(out / f"big_scorecard_{product}.parquet", index=False)
    gini_vars.to_parquet(out / f"gini_vars_{product}.parquet", index=False)
    stability_flags.to_parquet(out / f"bin_stability_flags_{product}.parquet", index=False)
    gini_time.to_parquet(out / f"gini_by_period_{product}.parquet", index=False)
    points_table.to_parquet(out / f"points_table_{product}.parquet", index=False)
    screen_report.to_parquet(out / f"feature_screen_report_{product}.parquet", index=False)
    with (out / f"pd_{product}.pkl").open("wb") as fh:
        pickle.dump(model_package, fh)
    woe_path = out / "woe_maps.pkl"
    existing = {}
    if woe_path.exists():
        with woe_path.open("rb") as fh:
            existing = pickle.load(fh)
    existing[product] = woe_maps
    with woe_path.open("wb") as fh:
        pickle.dump(existing, fh)
    cal_path = out / "calibration_params.json"
    cal_existing = {}
    if cal_path.exists():
        with cal_path.open(encoding="utf-8") as fh:
            cal_existing = json.load(fh)
    cal_existing[product] = calibration["params"]
    with cal_path.open("w", encoding="utf-8") as fh:
        json.dump(cal_existing, fh, indent=2)

In [ ]:
model_report = build_model_report(
    model_package, points_table, gini_time, calibration, cal_table, gini_vars
)
display_model_report(model_report)

save_product_artifacts(
    PRODUCT,
    "../data/06_models",
    big_scorecard,
    gini_vars,
    stability_flags,
    gini_time,
    model_package,
    points_table,
    screen_report,
    woe_maps,
    calibration,
)

gini_valid = model_package["metrics"]["gini_valid"]
print(f"PD {PRODUCT}: {len(model_package['features'])} features, valid Gini {gini_valid:.1%}")
print("Saved parquet/pkl for", PRODUCT)
print("When ins is done, set PRODUCT='css' and re-run §2–§6")

### Optional: full pipeline regression (skips §2 stability)

In [ ]:

# Optional regression pipeline — SKIPS §2 stability loop; use only after manual tuning.
# results = run_full_scorecard_pipeline(
#     "../data/04_feature/abt_app.parquet",
#     "../data/04_feature/decisions.parquet",
#     SCORECARD_PARAMS,
#     output_dir="../data/06_models",
# )
